# Recurrent Looped Transformer — a computational dissection

**Paper:** *Recurrent Looped Transformer*, Yifan Zhang, Jichen Feng, Shihan Qin — September 12, 2026.

We are not implementing this paper. We are **opening it up**: reconstructing the computation from first principles so that questions like *"what exactly is in the recurrent state at token t"*, *"what carries from t to t+1"*, *"what does the gate do numerically"* become things we **measure**, not things we assert.

---

### The hard constraint

**Pure Python.** No PyTorch, TensorFlow, JAX, Flax, Keras, HuggingFace, autograd — and **no NumPy**. Lists, floats, ints, dicts. The only import anywhere is `math`.

Everything is written inline in this notebook. There is no hidden module to jump to.

---

### Status

| Phase | What | State |
|---|---|---|
| **0** | Read the paper as a scientist | ✅ this notebook, §0 |
| **1** | Turn it into a computation graph | ✅ §1 |
| **2** | Build the mathematical primitives | ✅ §2, 67 invariants passing |
| 3 | Tiny RLT (d=8, V=10, L=1+1) | next |
| 4–9 | Full forward trace, shape tracker, trajectories, recurrence / cache / attention dissection | |
| 10–12 | Manual gradients, BPTT through the recurrence, parameter + gradient dissection | |
| 13–18 | Information flow, claim verification, ablation lab, write-up | |

Mirror repo (notes + a `.py` copy of §2): **[Maverick-Ansh/recurrent_looped_transformer_scratch](https://github.com/Maverick-Ansh/recurrent_looped_transformer_scratch)**

---

## ⚠️ Read this before anything else: **the paper has no experiments**

It says so three times:

- §1 — *"The report develops these mechanisms; **it does not report measured efficiency or scaling results**."*
- §3.2 — *"no reduced-prefill speedup is claimed."*
- §8 — *"The computational definitions are explicit; realized reasoning quality, hardware efficiency, and scaling behavior **require future validation**."*

Grep of the full 19-page text confirms it:

| searched | hits |
|---|---|
| `512`, `1365`, `4+4`, `accuracy`, `we train`, `we evaluate` | **0** |
| `parity` | 1 — and it is *"does not prove numerical kernel **parity**"* (§5.3), about kernel numerics, **not** the parity task |

There is no task, no dataset, no baseline, no results table, no hyperparameter appendix.

**So the config "width 512, FFN 1365, 4 heads, 8 layers, SWA 8, α=0.1, splits 4+4…8+0" is *not* from this paper.** It matches `C:\Users\ANSH\rlt-reproduce\rlt\config.py` — an earlier reproduction, i.e. a *previously chosen* experimental design. Recorded here so it never gets laundered into "the paper says".

**What this changes:** there is nothing to re-run. There are **propositions to verify** (§3.1, App. B.1) and a set of stated structural non-equivalences (§2.4, §2.6, App. B, App. C). Those are *exactly* what a pure-Python implementation can check **exactly** — we control every float. Any task or config in this notebook is **ours**, and is labelled as ours.

---

### Tagging convention, used everywhere below

- **[PAPER]** — stated in the paper, with section/equation.
- **[UNSPECIFIED]** — the paper does not say.
- **[ASSUMPTION]** — our choice, because we need a number to run code.

Paper facts and our choices are never mixed.

# §0 — Phase 0: the paper, extracted

## 0.1 Notation

| Symbol | Type | Meaning | §
|---|---|---|---|
| $x_{1:S}$ | token ids, $x_1=\text{BOS}$ | one independent sequence | 2.1 |
| $T$ | int | serving boundary — *"marks a serving boundary, **not a change in the conditional model**"* | 2.1 |
| $d$ | int | residual width | 2.1 |
| $L_E, L_D$ | int | encoder / decoder depth | 2.1 |
| $e_t \in \mathbb{R}^d$ | vector | encoder representation | 2.1 |
| $s_t \in \mathbb{R}^d$ | vector | recurrent decoder output | 2.1 |
| $C^D_t$ | per-layer (k,v) lists | retained decoder SWA KV at **every** layer | 2.1 |
| $H_t=(s_t, C^D_t)$ | pair | **complete decoder state** | 2.1 |
| $W \ge 1$ | int | SWA window — **includes the current token** | 2.1 |
| $G \in [1, L_D]$ | int | encoder-memory groups; $G{=}1$ shares, $G{=}L_D$ per-layer | 2.2 |
| $M^g_{\le t}$ | growing (k,v) set | encoder-derived cross-attention memory | 2.2 |
| $s_\star$ | vector | **learned** initial state, part of $\Theta$ | 2.3 |
| $\alpha$ | scalar | feedback scale | 2.5 |
| $\mu$ | dist | **behavior** policy (sampler) ≠ $p_\Theta$ | 5.3 |

⚠️ The paper overloads $W$: the scalar window **and** the matrices $W_g, W_s, W_V, W_o$. We always subscript matrices.

---

## 0.2 Every equation

**Encoder + memory**

$$e_{1:T} = E_\theta(x_{1:T}) \tag{2.1}$$

$$k^g_t = \mathcal{P}^g_K(e_t, t), \qquad v^g_t = W^g_V\,\mathrm{RMSNorm}_E(e_t), \qquad M^g_{\le t} = \{(k^g_j, v^g_j)\}_{j=1}^{t} \tag{2.2}$$

> [PAPER] *"The key map includes normalization, projection, and any positional transformation."* — so $\mathcal{P}_K$ = norm → linear → positional. Values get norm + linear but **no** positional op.
> [PAPER] *"This global memory depends on encoder representations, **not decoder states**."* ← the crucial asymmetry.

**The recurrence**

$$H_0 = (s_\star, \varnothing) \tag{2.3}$$
$$u_t = \mathrm{Merge}(e_t, s_{t-1}) \tag{2.4}$$
$$H_t = (s_t, C^D_t) = D_\phi(u_t;\, M_{\le t},\, C^D_{t-1},\, t), \quad t \ge 1 \tag{2.5}$$
$$p_\Theta(x_{t+1}\mid x_{1:t}) = \mathrm{softmax}\!\left(W_o\,\mathrm{RMSNorm}_o(s_t)\right)_{x_{t+1}} \tag{2.6}$$
$$H_T = F_T \circ F_{T-1} \circ \cdots \circ F_1(H_0) \tag{2.7}$$

> [PAPER] §2.3: *"**Neither component** of $H_T$ is reset at the serving boundary."*

**The merge (the gate)**

$$r_{t-1} = \mathrm{RMSNorm}_s(s_{t-1}) \tag{2.9}$$
$$g_t = \sigma\!\left(W_g[e_t; r_{t-1}] + b_g\right) \tag{2.10}$$
$$u_t = e_t + \alpha\, g_t \odot W_s r_{t-1} \tag{2.11}$$

with $W_g \in \mathbb{R}^{d\times 2d}$, $W_s \in \mathbb{R}^{d\times d}$, $g_t \in (0,1)^d$.

> ⚠️ **Read (2.11) again.** $W_s$ multiplies $r_{t-1}$ — the **normalized** state — not $s_{t-1}$. A trace that prints "$W_s s_{t-1}$" is printing a different quantity. We implement the paper and will print both.
> [PAPER] *"$\alpha$ controls the feedback scale. A modest nonzero initial feedback scale is a **candidate** initialization, **not an established stability prescription**."* — the paper declines to pick a value.

**The decoder block** — order is SWA → cross-attention → FFN, with $z^0_t = u_t$:

$$q^{D,\ell}_t = \mathcal{P}^{D,\ell}_Q(z^{\ell-1}_t, t) \tag{2.12}$$
$$k^{D,\ell}_t = \mathcal{P}^{D,\ell}_K(z^{\ell-1}_t, t), \qquad v^{D,\ell}_t = W^{D,\ell}_V \mathrm{RMSNorm}_{S,\ell}(z^{\ell-1}_t) \tag{2.13}$$
$$b^\ell_t = z^{\ell-1}_t + \mathrm{Attn}^D_\ell\!\left(q^{D,\ell}_t,\ \{(k^{D,\ell}_j, v^{D,\ell}_j)\}_{j=\max(1,\,t-W+1)}^{t}\right) \tag{2.14}$$
$$a^\ell_t = b^\ell_t + \mathrm{Attn}^M_\ell\!\left(\mathcal{P}^{M,\ell}_Q(b^\ell_t, t),\ M^{g(\ell)}_{\le t}\right) \tag{2.15}$$
$$z^\ell_t = a^\ell_t + \mathrm{FFN}_\ell\!\left(\mathrm{RMSNorm}_{D,\ell}(a^\ell_t)\right), \qquad s_t = z^{L_D}_t \tag{2.16}$$

> [PAPER] *"The attention operators include their output projections; query/key maps include their respective normalizations and positional transformations."*
> [PAPER] *"At each layer, current KV is formed **before** SWA, using the layer input, so current-position attention introduces **no circular dependency**."* — no fixpoint, no iteration.
> [PAPER] retention: *"After the update, retain positions $\max(1, t-W+2), \dots, t$ in $C^D_t$; this set is **empty for $W=1$**."*

**What "looped" means** — §2.6: the reference tied config sets $L_E = L_D = L$ and **shares** Q/K/V/O and FFN between encoder layer $\ell$ and decoder SWA layer $\ell$. Cross-attention gets separate query/output projections. So: **one backbone, two logical passes**, $2L$ block evaluations per token.

> [PAPER] *"This is parameter reuse with **different attention wiring**, not activation copying. No decoder output is identified with an encoder output."*

## 0.3 The four stores — and the fact that decides everything downstream

| # | Object | Shape | Bound | Evicts? | Depends on |
|---|---|---|---|---|---|
| 1 | $s_t$ | $[d]$ | one vector | overwritten | $x_{1:t}, \Theta$ |
| 2 | $C^D_t$ | $L_D \times (\le W{-}1) \times (k,v)$ | **bounded** | **yes** | $x_{1:t}, \Theta$ |
| 3 | $M^g_{\le t}$ | $G \times t \times (k,v)$ | unbounded | no | $x_{1:t}, \theta$ **only** |
| 4 | $C^E_t$ | encoder causal KV | unbounded | no | $x_{1:t}, \theta$ only |

> [PAPER] §4.2: *"Decoder layers read this memory and separately append decoder-derived KV to their bounded layerwise SWA caches. **These caches cannot be replaced by encoder memory.**"*

---

### ⚡ There are **two** recurrent channels, not one

Almost every mental model of this architecture has one backward arrow: $s_t \to s_{t+1}$. That is **wrong**, and Appendix B says so with a $2\times2$ Jacobian:

$$J_t = \frac{\partial H_t}{\partial H_{t-1}} = \begin{bmatrix} \dfrac{\partial s_t}{\partial s_{t-1}} & \dfrac{\partial s_t}{\partial C^D_{t-1}} \\[2ex] \dfrac{\partial C^D_t}{\partial s_{t-1}} & \dfrac{\partial C^D_t}{\partial C^D_{t-1}} \end{bmatrix} \tag{B.3}$$

$$\frac{\partial H_t}{\partial H_j} = J_t J_{t-1}\cdots J_{j+1}, \qquad j < t \tag{B.2}$$

> [PAPER] **"A product involving only $\partial s_t/\partial s_{t-1}$ generally misses paths through decoder KV. Normalization alone does not bound products of these Jacobians."**

So the state crossing the token boundary is $(s_t, C^D_t)$ — the vector **and** every decoder layer's sliding window.

```
   s_{t-1} ──────────────► MERGE ──► u_t ──► decoder ──► s_t ────────►  channel 1
                                       ▲         │
   C^D_{t-1} ──────────────────────────┘         └─────► C^D_t ───────►  channel 2
```

### 🔧 Design correction this forces on us

**Setting $\alpha = 0$ does *not* give a non-recurrent model.** It severs channel 1 (equation 2.11 collapses to $u_t = e_t$) and leaves channel 2 running at full strength.

A genuine non-recurrence control needs **both**:

$$\alpha = 0 \quad\textbf{and}\quad W = 1$$

because the paper states the retained set is *empty* at $W=1$. So the Phase-7 ablation is a **2×2 over $(\alpha, W)$**, not a single knob. (The earlier `rlt-reproduce` run used $\alpha=0$ alone as "the" recurrence ablation — that is the channel-1 ablation only.)

Similarly:
- **Zeroing $s_{t-1}$ does not erase history** — the window still holds $W-1$ decoder-derived KV entries.
- **Eviction is not forgetting.** [PAPER] App. B: *"Evicted entries can still influence later computation through states or retained activations that previously consumed them."*
- **Eviction is not a stop-gradient.** [PAPER] App. C: *"SWA eviction limits future direct access to old KV; **it is not itself a stop-gradient operation**. Full BPTT still differentiates computations that consumed those entries before eviction. The inference cache size therefore **does not bound** full-BPTT activation storage."*

---

## 0.4 The two propositions

**Prop 3.1 — invariance to the serving split.** Batched prefill + recurrent decoding ≡ fully incremental processing; *"Moving the prompt–response split does not change the conditional distribution for a fixed token history."* Proof is a one-line induction: both start at $H_0$; identical ops on identical inputs. *"The serving split never appears in the transition."*
→ In pure Python with one code path, the only way this fails is if **we** leak the split. That is what makes it a good test.

**Prop B.1 — causality.** $H_t$ depends only on $x_{1:t}$ and $\Theta$.
→ Checkable by perturbation: change $x_{t+1}$, assert $H_t$ is **bit-identical**. The cleanest experiment in the paper.

---

## 0.5 Claims extracted for Phase 14 — to be **tested**, not assumed

| ID | Claim | Source |
|---|---|---|
| C1 | State path composes $t$ transitions / $t\,L_D$ blocks; per-token count fixed at $L_E{+}L_D$ | §3.3, Fig 2 |
| C2 | Prompt and response use the same transition; moving $T$ changes nothing | Prop 3.1 |
| C3 | $H_t$ depends only on $x_{1:t}$ | Prop B.1 |
| C4 | **Both** $s$ and $C^D$ cross the boundary unreset | §2.3 |
| C5 | Encoder memory is prefix-restricted **per decoder position** | §2.4 |
| C6 | Cached states under old parameters are not current-policy states | §5.4, App C |
| C7 | Detaching $s$ alone is **not** full BPTT | App B, C |
| C8 | retention $[\max(1,t{-}W{+}2), t]$ + current == next read window $[\max(1,t{-}W{+}1), t{+}1]$ | §2.5 |
| C9 | An $\partial s/\partial s$-only product **misses** the KV paths | App B (B.3) |
| C10 | Gradient reach **>** cache reach | App C |
| C11 | $M$ never depends on decoder states | §2.2 |

C9 and C10 are the best targets: both are stated **without proof** and both are easy to get wrong in an implementation.

**C8 is already verified below in §2.5.**

---

## 0.6 What the paper does **not** specify

Every one of these needs a labelled choice from us.

| | Quantity | Paper's constraint | **[ASSUMPTION]** (tiny model) |
|---|---|---|---|
| A1 | $d$ | none | `8` |
| A2 | $V$ | none | `10` |
| A3 | $L_E, L_D$ | none (48+48 is *"illustrative"*, Fig 2) | `1, 1` |
| A4 | heads | **never mentioned at all** | `2` |
| A5 | $d_{ff}$ | none | `16` |
| A6 | FFN activation | none | GELU (both exact + tanh built) |
| A7 | $W$ | $W\ge1$ | `3` |
| A8 | $G$ | $1\le G\le L_D$ | `1` |
| A9 | $\alpha$ | *"modest nonzero"* | `0.1`, swept in Phase 7 |
| A10 | RMSNorm $\epsilon$ | none | `1e-5` |
| A11 | attention scale | none | $1/\sqrt{d_\text{head}}$ |

**Structural choices left open — implement *both* where practical:**

| | Choice | Paper's words | Plan |
|---|---|---|---|
| B1 | positional transform in $\mathcal{P}_Q,\mathcal{P}_K$ | *"any positional transformation"* | NoPE **and** RoPE; NoPE default so position doesn't confound state effects |
| B2 | tied vs untied $E/D$ | §2.6 tied is "reference"; untied *"preserves the complete-state recurrence"* | untied first (fewer confounds), tied as a flag |
| B3 | gate shape | vector $g_t$, or *"scalar gating"* | vector; scalar as a flag |
| B4 | $W_s$ rank | full, or *"low-rank"* | full |
| B5 | RMSNorm learned gain | unstated | with gain, init to 1 → no-gain variant **is** the init |

**Entirely absent:** optimizer, LR, schedule, batch size, init scheme, dropout, tokenizer, data, sequence length, $s_\star$ init. All become logged config fields so no result is ever read as "the paper's".

**Genuine ambiguities, flagged not hidden:**
1. $M_{\le t}$ must be **sliced per decoder position**, not just appended. §4.2: *"a faster kernel that reads future entries **changes the model**."*
2. Head split in cross-attention — [UNSPECIFIED]. [ASSUMPTION] same head count as SWA.
3. Does BOS get a decoder update? **Yes** — $t{=}1$ is BOS, produces $H_1$, predicts $x_2$. Consistent with (5.1) summing from $t{=}1$.
4. Is $\alpha$ trainable? [UNSPECIFIED]. [ASSUMPTION] fixed, so $\alpha{=}0$ is an exact clean ablation.

# §1 — Phase 1: the computation graph

Shapes below are for the **tiny config** of Phase 3, so every number is one you can print:

```
V = 10    d = 8    L_E = 1    L_D = 1    H = 2    d_head = 4
d_ff = 16    W = 3    G = 1    alpha = 0.1
```

All [ASSUMPTION] — see §0.6.

---

## 1.1 Encoder path (per token t)

```
x_t                                    int in [0,10)
  ↓ embedding lookup E_tok[x_t]
h_t^0                                  [8]    initial residual stream
  ↓ RMSNorm  →  W_Q/W_K/W_V  →  split heads
q,k,v                                  [2,4]  each
  ↓ append (k,v) to encoder cache C^E          ← STORE 4, grows forever
  ↓ causal attention over j = 1..t
score[h][j] = <q_t[h], k_j[h]> / sqrt(4)   [2,t]
  ↓ softmax over j
p[h][j]                                [2,t]  sums to 1 along j
  ↓ sum_j p[h][j] · v_j[h] → flatten → W_O
  ↓ residual add
b_t                                    [8]
  ↓ RMSNorm → FFN (8→16→8, GELU) → residual add
h_t^1  =  e_t                          [8]    ENCODER REPRESENTATION   (2.1)
```

**Meaning of $e_t$:** a causal, **non-recurrent** feature of $x_{1:t}$. It has *never seen a decoder state*. That asymmetry is what makes the encoder parallelizable (§4.1) and is what Prop B.1's proof leans on.

**Memory projection (2.2):**
```
e_t → RMSNorm_E → P_K^g (norm,linear,positional) → k_t^g   [2,4]
                → W_V^g  (no positional op)       → v_t^g   [2,4]
                → append to M^g                             ← STORE 3, unbounded
```

---

## 1.2 The merge — the entire recurrence for channel 1, in three lines

```
s_{t-1}                                [8]   (s_* at t=1)
  ↓ RMSNorm_s :  r_i = s_i / sqrt(mean(s²)+eps) · gain_i        (2.9)
r_{t-1}                                [8]

e_t, r_{t-1}
  ↓ concat                                                      (2.10)
[e_t ; r_{t-1}]                        [16] = [2d]
  ↓ W_g @ .  + b_g          W_g is [8,16]
gate preactivation                     [8]
  ↓ sigma elementwise
g_t                                    [8]   ∈ (0,1)^8   ← THE GATE

r_{t-1}
  ↓ W_s @ .                 W_s is [8,8]                        (2.11)
W_s r_{t-1}                            [8]
  ↓ ⊙ g_t
g_t ⊙ W_s r_{t-1}                      [8]
  ↓ × alpha
alpha · g_t ⊙ W_s r_{t-1}              [8]   ← FEEDBACK CONTRIBUTION
  ↓ + e_t
u_t                                    [8]   ← MERGED DECODER INPUT
```

$\alpha = 0$ kills the entire third block. $u_t = e_t$ exactly. $g_t$ is still computed and still depends on $r_{t-1}$, but is multiplied by zero.

---

## 1.3 Decoder block ($z^0_t = u_t$)

**A — causal SWA (2.12–2.14).** The step to watch:

```
z_t^{l-1}  →  P_Q (norm,W_Q,pos)  →  q   [2,4]
           →  P_K (norm,W_K,pos)  →  k   [2,4]
           →  RMSNorm_S → W_V     →  v   [2,4]        ← note: no positional op on v

C^D_{t-1}[l]        ≤ W-1 = 2 entries                 ← STORE 2
  ↓ read window = C^D_{t-1}[l] ++ [(k_t,v_t)]
read window         j ∈ [max(1,t-W+1), t], ≤ 3 entries

  ↓ score[h][j] = <q[h],k_j[h]>/sqrt(4) → softmax over j
  (no mask needed — THE WINDOW *IS* THE MASK)
  ↓ sum_j p·v → flatten → W_O → residual add
b_t^l = z_t^{l-1} + swa_out            [8]                      (2.14)

  ↓ retain [max(1,t-W+2), t]  (drop oldest if full)
C^D_t[l]            ≤ W-1 = 2 entries
```

**B — cross-attention to encoder memory (2.15).**

```
b_t^l → P_Q^M → cross query    [2,4]
M_{<=t}^{g(l)}                 [t][2,4]×2    ← PREFIX-RESTRICTED, sliced to t
  ↓ scores over j=1..t → softmax → weighted V → W_O^M → residual add
a_t^l = b_t^l + cross_out      [8]                              (2.15)
```

The prefix restriction is a **model property, not an optimization**. §2.4: *"At decoder position t, attention is restricted to $M_{\le t}$ even though the entire prompt memory is available."*

**C — FFN (2.16).**
```
a_t^l → RMSNorm_D → W_1 [16,8] → GELU → W_2 [8,16] → residual add
z_t^l                          [8]
                     s_t = z_t^{L_D}    ← THE RECURRENT STATE
```

**Readout (2.6).** `s_t → RMSNorm_o → W_o [10,8] → logits [10] → softmax → p(x_{t+1}|x_{1:t})`

---

## 1.4 The recurrent path, unrolled — **both** channels, nothing hidden

```
H_0 = (s_*, ∅)
 │  F_1 : u_1 = Merge(e_1, s_*)   → decoder →
 ▼
H_1 = (s_1, C^D_1)     C^D_1[l] = [ (k_1,v_1) ]                     W=3
 │  F_2 : u_2 = Merge(e_2, s_1)   → decoder →
 ▼
H_2 = (s_2, C^D_2)     C^D_2[l] = [ (k_1,v_1), (k_2,v_2) ]
 │  F_3 : u_3 = Merge(e_3, s_2)   → decoder →
 ▼
H_3 = (s_3, C^D_3)     C^D_3[l] = [ (k_2,v_2), (k_3,v_3) ]   ← (k_1,v_1) EVICTED
 │  F_4 : u_4 = Merge(e_4, s_3)   → decoder →
 ▼
H_4 = (s_4, C^D_4)     C^D_4[l] = [ (k_3,v_3), (k_4,v_4) ]
```

### Where does $x_1$ survive at $t=4$? Three routes — enumerating them **is** Phase 13.

1. ❌ **Not** in $C^D_4$ — $(k_1,v_1)$ was evicted at $t=3$.
2. ✅ In $M_{\le 4}$ — encoder memory never evicts.
3. ✅ In $s_4$, **indirectly**: $(k_1,v_1)$ was read by SWA at $t=1,2,3$, shaping $s_1,s_2,s_3$; and $s_3$ feeds $u_4$.

Route 3 is exactly App. C's *"evicted entries can still influence later computation through states … that previously consumed them."* Route 3 is **also** why eviction is not a stop-gradient (C10).

---

## 1.5 Block-count accounting (claim C1)

```
per token:      L_E + L_D  blocks          ← FIXED, independent of t
along chain:    t · L_D    decoder blocks  ← GROWS with t
```

Paper's illustrative $L_E{=}L_D{=}48$: 96 blocks/token, $48t$ along the chain — exactly Fig. 2. Our tiny config: 2 blocks/token, $t$ along the chain. Phase 14 tests this by **counting actual block invocations**, not by re-deriving the formula.

# §2 — Phase 2: the mathematical primitives

Everything below is written from scratch. The **only** import in this entire section is `math`.

Two rules held throughout:

1. **`attention()` returns `(out, scores, probs)` — all three.** A function that returned only `out` would be exactly the opaque helper this project exists to avoid. Phase 9 inspects every intermediate, so the intermediates must escape.
2. **Initialization has no hidden machinery.** A 10-line linear congruential generator + Box–Muller, rather than inheriting a Mersenne Twister — so the path from bits to weights is readable, and the same seed gives the same weights on any platform.

Data types, fixed here for the whole project:

| name | Python type | shape |
|---|---|---|
| scalar | `float` | |
| vector | `list[float]` | `d` |
| matrix | `list[list[float]]` | `[d_out][d_in]` — paper uses **column vectors**, so `matvec(M,x)` is $Mx$ |
| heads | `list[list[float]]` | `[H][d_head]` |
| kv entry | `(pos, k, v)` | |
| swa cache | `list[list[kv]]` | `[layer][slot]` |
| attn scores | `list[list[float]]` | `[H][n_keys]` |

Run the cells in order — each builds on the previous.

In [ ]:
# =============================================================================
# 2.1  THE PURITY GUARD, then vectors, matrices, reductions
# =============================================================================
import math          # <-- the ONLY import in all of section 2
import sys

# A guard on sys.modules would measure the HOST, not this code: Colab pre-imports
# numpy before we ever run. So we snapshot what is ALREADY loaded, and later assert
# that *we* never bind any of it.
_PRELOADED = set(sys.modules)
BANNED = {"numpy", "torch", "tensorflow", "jax", "flax", "keras",
          "transformers", "scipy", "sklearn", "pandas"}

def assert_pure(namespace):
    """Fail if any banned library leaked into OUR namespace (not the host's)."""
    leaked = []
    for name, obj in list(namespace.items()):
        mod = getattr(obj, "__module__", None) or getattr(obj, "__name__", "")
        root = str(mod).split(".")[0]
        if root in BANNED:
            leaked.append((name, root))
    assert not leaked, f"PURITY VIOLATION: {leaked}"
    return (f"pure: none of {sorted(BANNED)} is bound in this notebook "
            f"(host had {len(_PRELOADED)} modules preloaded -- irrelevant, that is Colab's)")

NEG_INF = float("-inf")

# ------------------------------------------------------------------ vectors --
def vadd(a, b):
    """(a+b)_i = a_i + b_i"""
    assert len(a) == len(b), f"vadd mismatch {len(a)} vs {len(b)}"
    return [a[i] + b[i] for i in range(len(a))]

def vsub(a, b):
    """(a-b)_i = a_i - b_i"""
    assert len(a) == len(b), f"vsub mismatch {len(a)} vs {len(b)}"
    return [a[i] - b[i] for i in range(len(a))]

def vscale(c, a):
    """(c*a)_i = c*a_i"""
    return [c * a[i] for i in range(len(a))]

def vmul(a, b):
    """(a (*) b)_i = a_i*b_i  -- the elementwise product of eq (2.11)."""
    assert len(a) == len(b), f"vmul mismatch {len(a)} vs {len(b)}"
    return [a[i] * b[i] for i in range(len(a))]

def vneg(a):        return [-x for x in a]
def vzeros(n):      return [0.0] * n

def vsum(a):
    """sum_i a_i -- written as a loop, not sum(), so the accumulation is visible."""
    tot = 0.0
    for x in a:
        tot += x
    return tot

def dot(a, b):
    """<a,b> = sum_i a_i*b_i"""
    assert len(a) == len(b), f"dot mismatch {len(a)} vs {len(b)}"
    tot = 0.0
    for i in range(len(a)):
        tot += a[i] * b[i]
    return tot

# ----------------------------------------------------------------- matrices --
def matvec(M, x):
    """y = Mx,  y_i = sum_j M_ij x_j.   M is [d_out][d_in], x is [d_in]."""
    assert len(M) > 0 and len(M[0]) == len(x), \
        f"matvec mismatch: M=[{len(M)},{len(M[0])}] x=[{len(x)}]"
    return [dot(M[i], x) for i in range(len(M))]

def matmul(A, B):
    """C = AB,  C_ij = sum_k A_ik B_kj."""
    assert len(A[0]) == len(B), f"matmul mismatch: A cols {len(A[0])} vs B rows {len(B)}"
    n, k, m = len(A), len(B), len(B[0])
    C = [[0.0] * m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            acc = 0.0
            for p in range(k):
                acc += A[i][p] * B[p][j]
            C[i][j] = acc
    return C

def transpose(M):   return [[M[i][j] for i in range(len(M))] for j in range(len(M[0]))]
def mzeros(r, c):   return [[0.0] * c for _ in range(r)]

# --------------------------------------------------- reductions / statistics --
def mean(a):  return vsum(a) / len(a)

def var(a):
    m = mean(a)
    return vsum([(x - m) ** 2 for x in a]) / len(a)

def rms(a):
    """rms(a) = sqrt( (1/n) sum_i a_i^2 )

    This is the UNCENTERED second moment -- it does NOT subtract the mean.
    That is precisely what separates RMSNorm from LayerNorm.
    """
    return math.sqrt(vsum([x * x for x in a]) / len(a))

def l2(a):
    """||a||_2 = sqrt(sum_i a_i^2).   Relation: ||a|| = sqrt(n) * rms(a)."""
    return math.sqrt(vsum([x * x for x in a]))

def vmin(a):
    m = a[0]
    for x in a:
        if x < m: m = x
    return m

def vmax(a):
    m = a[0]
    for x in a:
        if x > m: m = x
    return m


print(assert_pure(globals()))
print()
u = [3.0, -1.0, 0.0, 4.0]
w = [1.0, 2.0, -2.0, 0.5]
print(f"u          = {u}")
print(f"w          = {w}")
print(f"u + w      = {vadd(u, w)}")
print(f"u (*) w    = {vmul(u, w)}        <- eq (2.11) uses this")
print(f"<u,w>      = {dot(u, w)}")
print(f"mean(u)    = {mean(u)}")
print(f"rms(u)     = {rms(u):.6f}   (uncentered -- mean is NOT subtracted)")
print(f"l2(u)      = {l2(u):.6f}   == sqrt(4)*rms(u) = {math.sqrt(4)*rms(u):.6f}")
M = [[1.0, 0.0, 2.0, 0.0], [0.0, 1.0, 0.0, -1.0]]
print(f"\nM ({len(M)}x{len(M[0])}) @ u = {matvec(M, u)}    <- row i is <M[i], u>")

In [ ]:
# =============================================================================
# 2.2  NONLINEARITIES  -- softmax, log-softmax, sigma (eq 2.10), GELU
# =============================================================================

def softmax(z):
    """p_i = exp(z_i) / sum_j exp(z_j)

    Computed as exp(z_i - max z) / sum_j exp(z_j - max z). Algebraically
    IDENTICAL (the exp(max) factor cancels top and bottom) but it cannot
    overflow. Entries equal to -inf (masked positions) map to exactly 0.0.
    """
    m = vmax(z)
    if m == NEG_INF:
        raise ValueError("softmax over an all-masked row: every logit is -inf")
    exps = [0.0 if x == NEG_INF else math.exp(x - m) for x in z]
    denom = vsum(exps)
    return [e / denom for e in exps]


def log_softmax(z):
    """log p_i = z_i - log sum_j exp(z_j)

    Kept separate from log(softmax(z)): taking the log of a tiny probability
    loses precision, this form does not. The Phase-10 training loss uses it.
    """
    m = vmax(z)
    if m == NEG_INF:
        raise ValueError("log_softmax over an all-masked row")
    shifted = [(x - m) if x != NEG_INF else NEG_INF for x in z]
    lse = math.log(vsum([math.exp(x) if x != NEG_INF else 0.0 for x in shifted]))
    return [(x - lse) if x != NEG_INF else NEG_INF for x in shifted]


def sigmoid(x):
    """sigma(x) = 1 / (1 + exp(-x))   -- the 'sigma' of eq (2.10).

    Branch on the sign so neither exp() overflows:
        x >= 0 :  1 / (1 + exp(-x))
        x <  0 :  exp(x) / (1 + exp(x))      same value, multiplied through by exp(x)
    """
    if x >= 0.0:
        return 1.0 / (1.0 + math.exp(-x))
    e = math.exp(x)
    return e / (1.0 + e)


def gelu_exact(x):
    """GELU(x) = x * Phi(x) = x * 0.5 * (1 + erf(x/sqrt(2)))    [Hendrycks & Gimpel]"""
    return x * 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def gelu_tanh(x):
    """GELU(x) ~= 0.5x(1 + tanh( sqrt(2/pi)(x + 0.044715 x^3) ))  -- tanh approximation.

    [UNSPECIFIED] the paper writes only 'FFN_l' and never names an activation.
    Both forms are here so the choice is visible and swappable (see 0.6, A6).
    """
    c = math.sqrt(2.0 / math.pi)
    return 0.5 * x * (1.0 + math.tanh(c * (x + 0.044715 * x ** 3)))


def apply_mask(scores_row, mask_row):
    """Set disallowed logits to -inf so softmax sends them to exactly 0."""
    assert len(scores_row) == len(mask_row)
    return [scores_row[j] if mask_row[j] else NEG_INF for j in range(len(scores_row))]


# ------------------------------------------------------------------ numbers --
z = [1.0, -2.0, 0.5, 3.0, -0.25]
p = softmax(z)
print("softmax")
print(f"  z            = {z}")
print(f"  p            = {[round(v,6) for v in p]}")
print(f"  sum p        = {vsum(p):.15f}")
print(f"  softmax(z+17.3) == softmax(z)?  max|diff| = "
      f"{max(abs(a-b) for a,b in zip(softmax([v+17.3 for v in z]), p)):.3e}   (shift invariance)")

big = [1000.0, 999.0, 1001.0]
print(f"\n  logits of 1e3 -> p = {[round(v,6) for v in softmax(big)]}   "
      f"(naive exp(1000) would be inf)")

masked = apply_mask([1.0, 2.0, 3.0], [True, False, True])
print(f"  masked row {masked} -> p = {[round(v,6) for v in softmax(masked)]}   "
      f"(masked entry is EXACTLY 0.0)")

print("\nsigma  -- eq (2.10) squashes the gate preactivation into (0,1)")
for x in [-1000, -6, -2, 0, 2, 6, 1000]:
    print(f"  sigma({x:>6}) = {sigmoid(x):.9f}")

print("\nGELU")
worst = max(abs(gelu_exact(-4 + 0.05*i) - gelu_tanh(-4 + 0.05*i)) for i in range(161))
print(f"  gelu(0)                 = {gelu_exact(0.0)}")
print(f"  gelu(-0.5)              = {gelu_exact(-0.5):+.6f}   <- NOT monotone, it dips below 0")
print(f"  max|exact - tanh| on [-4,4] = {worst:.3e}")

In [ ]:
# =============================================================================
# 2.3  RMSNORM  -- eq (2.9), and the seven other norm sites in the model
# =============================================================================

def rmsnorm_raw(x, eps=1e-5):
    """The normalization alone, no learned gain:

        r_i = x_i / sqrt( mean(x^2) + eps )
            = x_i / sqrt( (1/n) sum_j x_j^2 + eps )

    eps sits INSIDE the sqrt (the usual convention). The mean is over the
    SQUARES -- the vector's own mean is never subtracted.
    """
    ms = vsum([v * v for v in x]) / len(x)
    return [v / math.sqrt(ms + eps) for v in x]


def rmsnorm(x, gain=None, eps=1e-5):
    """Full RMSNorm:   out_i = gain_i * x_i / sqrt(mean(x^2) + eps)

    [UNSPECIFIED] whether the paper's RMSNorm carries a learned gain. We include
    one, initialized to all-ones, so the no-gain variant IS the initialization
    (see 0.6, B5).
    """
    r = rmsnorm_raw(x, eps)
    if gain is None:
        return r
    assert len(gain) == len(x), f"gain {len(gain)} != input {len(x)}"
    return vmul(gain, r)


# ------------------------------------------------------------------ numbers --
v = [2.0, -1.0, 0.5, 3.0, -0.25, 1.5, -2.0, 0.75]

r0 = rmsnorm_raw(v, eps=0.0)
print("with eps = 0")
print(f"  x                    = {v}")
print(f"  rmsnorm(x)           = {[round(q,6) for q in r0]}")
print(f"  rms(output)          = {rms(r0):.15f}      <- exactly 1")
print(f"  mean(output)         = {mean(r0):+.6f}      <- NOT 0: RMSNorm does not centre")
print(f"  rmsnorm(37x) == rmsnorm(x)?  max|diff| = "
      f"{max(abs(a-b) for a,b in zip(rmsnorm_raw(vscale(37.0, v), eps=0.0), r0)):.3e}"
      f"   <- scale invariant")

print("\n" + "="*72)
print("FINDING: eps breaks scale invariance, and it breaks it exactly where a")
print("         decaying recurrent state lives.")
print("="*72)
print(f"  {'scale c':>10} {'rms(c*x)':>12} {'rms(rmsnorm(c*x))':>20}   eps=1e-5")
for c in [1e2, 1e0, 1e-1, 1e-2, 1e-3, 1e-4]:
    xc = vscale(c, v)
    print(f"  {c:>10.0e} {rms(xc):>12.3e} {rms(rmsnorm(xc, eps=1e-5)):>20.6f}")

print("""
RMSNorm is usually described as scale-invariant. At eps = 0 it is. At eps = 1e-5
it is not, and the failure is severe precisely when ||x|| is small -- which is the
regime a CONTRACTING recurrent state s_t lives in.

So a state that decays toward zero does NOT get renormalized back to unit RMS by
eq (2.9). It keeps shrinking THROUGH the norm, and r_{t-1} shrinks with it, which
scales down the whole feedback term  alpha * g_t (*) W_s r_{t-1}  in eq (2.11).

This is a concrete mechanism behind the paper's own caveat, section 3.3:
    "Gates, contraction, and learned projections may suppress the practical
     contribution of long paths; structural depth alone is not a reasoning
     guarantee."

Phase 6 measures whether ||s_t|| actually enters this regime. Noted now, not
assumed either way.""")

In [ ]:
# =============================================================================
# 2.4  SHAPE PLUMBING + MASKS, and the SWA window identity (claim C8)
# =============================================================================

def concat(a, b):
    """[a ; b] -- the concatenation of eq (2.10), giving length 2d."""
    return list(a) + list(b)

def slice_(a, start, stop):
    """a[start:stop] -- explicit, because Phase 8 slices caches constantly."""
    return a[start:stop]

def split_heads(x, n_heads):
    """[d] -> [H][d_head], contiguous. Head h owns dims [h*dh, (h+1)*dh).

    This is a VIEW decision, not mathematics -- the standard contiguous
    convention, chosen so merge_heads(split_heads(x,H)) == x exactly.
    """
    d = len(x)
    assert d % n_heads == 0, f"d={d} not divisible by n_heads={n_heads}"
    dh = d // n_heads
    return [x[h*dh:(h+1)*dh] for h in range(n_heads)]

def merge_heads(heads):
    """[H][d_head] -> [d], the exact inverse of split_heads."""
    out = []
    for h in heads:
        out.extend(h)
    return out

def causal_mask(T):
    """mask[i][j] = True iff query i may attend to key j.  Causal: j <= i."""
    return [[j <= i for j in range(T)] for i in range(T)]

def sliding_window_mask(T, W):
    """Causal AND within W positions INCLUDING the current one:

        mask[i][j] = True  iff  (i - W + 1) <= j <= i

    The paper (2.1) is explicit that W includes the current token, so W=1 means
    'attend to yourself only'.
    """
    assert W >= 1, f"window W must be >= 1, got {W}"
    return [[(j <= i) and (j >= i - W + 1) for j in range(T)] for i in range(T)]


# ------------------------------------------------------------------ numbers --
y = list(range(12))
y = [float(q) for q in y]
print(f"split_heads(0..11, H=3) = {split_heads(y, 3)}")
print(f"round-trips for H in 1,2,3,4,6,12: "
      f"{all(merge_heads(split_heads(y, H)) == y for H in [1,2,3,4,6,12])}")

def show_mask(m, title):
    print(f"\n  {title}")
    print("        j: " + " ".join(str(j) for j in range(len(m[0]))))
    for i, row in enumerate(m):
        print(f"      i={i}:  " + " ".join("T" if q else "." for q in row))

show_mask(causal_mask(5), "causal_mask(5)")
show_mask(sliding_window_mask(6, 3), "sliding_window_mask(6, W=3)  <- position 0 leaves at i=3")
show_mask(sliding_window_mask(5, 1), "sliding_window_mask(5, W=1)  <- diagonal only; NO history at all")

print("\n" + "="*72)
print("CLAIM C8  (paper section 2.5) -- the one that looks like an off-by-one")
print("="*72)
print("""The paper states TWO different intervals:

    read window at t   :  j in [ max(1, t-W+1), t ]     eq (2.14)
    retain after t     :  j in [ max(1, t-W+2), t ]     'After the update, retain...'

They differ by one. That is easy to misread as a bug. The claim to test is that
they are exactly consistent:

    retained(t)  UNION  {t+1}   ==   read_window(t+1)

no slack, no gap. Checking it for W in {1,2,3,8}, all t (0-indexed here):""")

c8_ok, rows = True, []
for W in [1, 2, 3, 8]:
    T = 12
    m = sliding_window_mask(T, W)
    for t in range(T - 1):
        read_t    = {j for j in range(T) if m[t][j]}
        retained  = {j for j in read_t if j >= t - W + 2}
        read_next = {j for j in range(T) if m[t+1][j]}
        if retained | {t+1} != read_next:
            c8_ok = False
            rows.append(f"    MISMATCH W={W} t={t}")
        if W == 3 and t < 6:
            rows.append(f"    W=3 t={t}:  read={sorted(read_t)!s:<12} "
                        f"retained={sorted(retained)!s:<10} -> next read={sorted(read_next)}")
print()
print("\n".join(rows))
print(f"\n  |retained| <= W-1 always:  "
      f"{all(len({j for j in range(T) if sliding_window_mask(T,W)[t][j] and j >= t-W+2}) <= W-1 for W in [1,2,3,8] for t in range(11))}")
print(f"  at W=1 the retained set is EMPTY (paper says so explicitly):  "
      f"{ {j for j in range(12) if sliding_window_mask(12,1)[5][j] and j >= 5-1+2} == set() }")
print(f"\n  ==> C8 VERIFIED: {c8_ok}")

In [ ]:
# =============================================================================
# 2.5  ATTENTION  -- the complete chain, nothing collapsed into one call
# =============================================================================

def attention(q, keys, values, scale=None):
    """Single-head scaled dot-product attention over an EXPLICIT key/value list.

        score_j = <q, k_j> / sqrt(d_head)
        p_j     = softmax(score)_j
        out     = sum_j p_j * v_j

    Returns (out, scores, probs) -- ALL THREE. A function returning only `out`
    would be exactly the opaque helper this project exists to avoid; Phase 9
    inspects every intermediate, so every intermediate must escape.

    For SWA the caller passes ONLY the window, so no mask is needed:
    the window IS the mask.
    """
    assert len(keys) == len(values), f"{len(keys)} keys vs {len(values)} values"
    assert len(keys) > 0, "attention over an empty key set"
    if scale is None:
        scale = 1.0 / math.sqrt(len(q))

    scores = [dot(q, k) * scale for k in keys]
    probs  = softmax(scores)

    out = vzeros(len(values[0]))
    for j in range(len(values)):
        out = vadd(out, vscale(probs[j], values[j]))
    return out, scores, probs


def multihead_attention(q_heads, k_heads_list, v_heads_list, scale=None):
    """Run `attention` per head, concatenate.

        q_heads       [H][d_head]
        k_heads_list  [n_keys][H][d_head]        one entry per key position
        v_heads_list  [n_keys][H][d_head]

    Returns (out [d], scores [H][n_keys], probs [H][n_keys]). Stacking the
    per-query scores gives the [heads, query_pos, key_pos] tensor of Phase 9.
    """
    H, n = len(q_heads), len(k_heads_list)
    assert n == len(v_heads_list) and n > 0
    out_heads, all_scores, all_probs = [], [], []
    for h in range(H):
        keys_h = [k_heads_list[j][h] for j in range(n)]
        vals_h = [v_heads_list[j][h] for j in range(n)]
        o, sc, pr = attention(q_heads[h], keys_h, vals_h, scale)
        out_heads.append(o); all_scores.append(sc); all_probs.append(pr)
    return merge_heads(out_heads), all_scores, all_probs


# ------------------------------------------------------------------ numbers --
# A worked example, printed as the full chain rather than as one number.
q    = [1.0, 0.0, 0.5, -0.5]
keys = [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.5, 0.5, 0.5, 0.5], [2.0, 0.0, 1.0, -1.0]]
vals = [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]]

out, scores, probs = attention(q, keys, vals)
dh = len(q)

print(f"q  = {q}          d_head = {dh},  scale = 1/sqrt({dh}) = {1/math.sqrt(dh):.6f}\n")
print("  j   k_j                        <q,k_j>    score=<q,k_j>/sqrt(d)   p_j")
for j in range(len(keys)):
    print(f"  {j}   {str(keys[j]):<26} {dot(q,keys[j]):+7.3f}   {scores[j]:+12.6f}        {probs[j]:.6f}")
print(f"\n  sum_j p_j = {vsum(probs):.15f}")

print("\n  out_i = sum_j p_j * v_j[i]:")
for i in range(4):
    terms = " + ".join(f"{probs[j]:.4f}*{vals[j][i]:.0f}" for j in range(len(vals)))
    print(f"    out[{i}] = {terms} = {out[i]:.6f}")
print(f"\n  out = {[round(o,6) for o in out]}")

# Invariants that catch sign/index errors a 'sums to 1' check would miss.
hull = all(min(vl[i] for vl in vals) - 1e-12 <= out[i] <= max(vl[i] for vl in vals) + 1e-12
           for i in range(4))
print(f"\n  output inside the convex hull of the values?  {hull}")
print(f"  identical keys -> uniform?  {[round(x,6) for x in attention(q,[keys[0]]*4,vals)[2]]}")
print(f"  one dominant score -> one-hot?  "
      f"{[round(x,9) for x in attention(q,[vscale(80.0,q),keys[1],keys[2]],vals[:3])[2]]}")

print("\nmulti-head, H=2 on d=8 (heads must NOT mix):")
q2 = [1.0,0.0,0.5,-0.5, 0.0,1.0,-1.0,0.25]
k2 = [[1.0,0,0,0, 0,0,0,0],[0,1.0,0,0, 0,0,1.0,0],[0,0,1.0,0, 1.0,0,0,0]]
v2 = [[1.0,0,0,0, 0,0,0,0],[0,1.0,0,0, 0,1.0,0,0],[0,0,1.0,0, 0,0,1.0,0]]
mo, ms, mp = multihead_attention(split_heads(q2,2), [split_heads(k,2) for k in k2],
                                                    [split_heads(v,2) for v in v2])
print(f"  scores shape [H][n_keys] = ({len(ms)}, {len(ms[0])})")
for h in range(2):
    print(f"  head {h}: scores={[round(x,4) for x in ms[h]]}  probs={[round(x,4) for x in mp[h]]}"
          f"  sum={vsum(mp[h]):.12f}")
h0 = attention(split_heads(q2,2)[0], [split_heads(k,2)[0] for k in k2],
                                     [split_heads(v,2)[0] for v in v2])[0]
print(f"  head-0 output computed alone == first half of merged output?  {mo[:4] == h0}")
print(f"  scale is 1/sqrt(d_head)=1/2, not 1/sqrt(d):  "
      f"{abs(ms[0][0] - dot(split_heads(q2,2)[0], split_heads(k2[0],2)[0])/2.0) < 1e-15}")

In [ ]:
# =============================================================================
# 2.6  RANDOM INITIALIZATION + INSPECTION HELPERS (seed of the Phase 5 tracker)
# =============================================================================

class LCG:
    """A linear congruential generator, so initialization has no hidden machinery.

        state <- (a*state + c) mod m      Numerical Recipes constants
        a = 1664525,  c = 1013904223,  m = 2^32

    Why not `random`? Reproducibility here should be readable in ten lines rather
    than inherited from a Mersenne Twister. Same seed -> same weights on any
    platform, any Python version.

    LCG low bits are famously poor, so `uniform` uses only the top 24 bits.
    """
    A, C, M = 1664525, 1013904223, 2 ** 32

    def __init__(self, seed=0):
        self.state = seed % self.M

    def next_u32(self):
        self.state = (self.A * self.state + self.C) % self.M
        return self.state

    def uniform(self):
        """Uniform on the OPEN interval (0,1) -- never exactly 0, which would
        make log(u) in Box-Muller blow up."""
        return ((self.next_u32() >> 8) + 0.5) / (2 ** 24)

    def normal(self):
        """Box-Muller:  z = sqrt(-2 ln u1) * cos(2*pi*u2)  is N(0,1)
        for u1,u2 iid Uniform(0,1). We discard the sin(...) partner for clarity."""
        u1, u2 = self.uniform(), self.uniform()
        return math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)


def randn_vector(n, rng, std=1.0):
    return [rng.normal() * std for _ in range(n)]

def randn_matrix(rows, cols, rng, std=1.0):
    """[UNSPECIFIED] the paper gives NO initialization scheme at all. Callers pass
    `std` explicitly (e.g. 1/sqrt(fan_in)) so the choice always lives at the call
    site and is never buried here."""
    return [[rng.normal() * std for _ in range(cols)] for _ in range(rows)]


# ------------------------------------------------ inspection (Phase 5 seed) --
def shape(x):
    """Recursive shape of nested lists, as a tuple. shape(3.0) == ()."""
    if isinstance(x, (int, float)):   return ()
    if isinstance(x, (list, tuple)):  return (0,) if len(x) == 0 else (len(x),) + shape(x[0])
    raise TypeError(f"shape() does not handle {type(x)}")

def _flatten(x):
    if isinstance(x, (int, float)): return [float(x)]
    out = []
    for item in x: out.extend(_flatten(item))
    return out

def summarize(name, x):
    """The Phase-5 record for one value: name, shape, dtype, min, max, mean, rms, l2.
    Returns a DICT, not a string -- trajectory.json wants raw numbers; formatting
    belongs in the tracer, not here."""
    flat = _flatten(x)
    finite = [v for v in flat if v != NEG_INF and v == v]
    rec = {"name": name, "shape": list(shape(x)), "dtype": "float",
           "n": len(flat), "n_masked": len(flat) - len(finite)}
    if finite:
        rec.update({"min": vmin(finite), "max": vmax(finite), "mean": mean(finite),
                    "rms": rms(finite), "l2": l2(finite)})
    return rec

def fmt(rec):
    """One-line rendering of a summarize() record."""
    if "min" not in rec:
        return f"{rec['name']:<22} shape={str(tuple(rec['shape'])):<10} (all masked)"
    return (f"{rec['name']:<22} shape={str(tuple(rec['shape'])):<10} "
            f"min={rec['min']:+9.4f} max={rec['max']:+9.4f} "
            f"mean={rec['mean']:+9.4f} rms={rec['rms']:8.4f} l2={rec['l2']:8.4f}")

def allclose(a, b, atol=1e-9, rtol=1e-7):
    """Elementwise |a-b| <= atol + rtol*|b| over arbitrarily nested lists.
    Returns (ok, max_abs_diff, index_of_worst)."""
    fa, fb = _flatten(a), _flatten(b)
    assert len(fa) == len(fb), f"allclose shape mismatch: {len(fa)} vs {len(fb)}"
    worst, worst_i = 0.0, -1
    for i in range(len(fa)):
        d = abs(fa[i] - fb[i])
        if d > worst: worst, worst_i = d, i
        if d > atol + rtol * abs(fb[i]): return False, worst, worst_i
    return True, worst, worst_i


# ------------------------------------------------------------------ numbers --
# NOTE: draw from ONE generator. [LCG(99).normal() for _ in range(5)] rebuilds the
# generator every iteration and returns the SAME number five times -- a vacuous
# test that passes. This bit the first draft of the test suite.
ga, gb, gc = LCG(99), LCG(99), LCG(100)
sa = [ga.normal() for _ in range(5)]
sb = [gb.normal() for _ in range(5)]
sc = [gc.normal() for _ in range(5)]
print(f"LCG(99)  stream = {[round(x,6) for x in sa]}")
print(f"LCG(99)  again  = {[round(x,6) for x in sb]}    identical? {sa == sb}")
print(f"LCG(100) stream = {[round(x,6) for x in sc]}    differs?   {sa != sc}")
print(f"stream does not repeat itself: {len(set(sa)) == 5}")

g = LCG(2024); N = 20000
samples = [g.normal() for _ in range(N)]
print(f"\nBox-Muller over N={N}:  mean={mean(samples):+.4f} (want 0)   "
      f"std={math.sqrt(var(samples)):.4f} (want 1)")

g5 = LCG(5); us = [g5.uniform() for _ in range(1000)]
bins = [0]*10
for t in us: bins[min(9, int(t*10))] += 1
print(f"uniform: min={min(us):.6f} max={max(us):.6f} distinct={len(set(us))}/1000")
print(f"         10 equal bins of 1000 draws = {bins}   (want ~100 each)")

print("\nsummarize() -- the Phase 5 record:")
rng = LCG(7)
print("  " + fmt(summarize("s_t",              randn_vector(8, rng))))
print("  " + fmt(summarize("attn_scores[H,n]", [randn_vector(5, rng) for _ in range(2)])))
print("  " + fmt(summarize("W_g",              randn_matrix(8, 16, rng, std=1/math.sqrt(16)))))
print(f"\n  masked entries counted: {summarize('m',[1.0,NEG_INF,2.0])['n_masked']}")
print(f"  allclose([1,2],[1,2.5]) -> {allclose([1.0,2.0],[1.0,2.5])}   (ok, worst, index)")

In [ ]:
# =============================================================================
# 2.7  THE INVARIANT SUITE -- every primitive above, checked
# =============================================================================
# No pytest, no unittest: a 10-line harness, zero dependencies.

_PASS, _FAIL = [], []

def check(name, cond, detail=""):
    (_PASS if cond else _FAIL).append(name)
    print(("  ok   " if cond else "  FAIL ") + name + (f"   {detail}" if detail else ""))

def close(a, b, atol=1e-9, rtol=1e-7):
    ok, worst, _ = allclose(a, b, atol, rtol)
    return ok, f"max|diff|={worst:.3e}"

def section(t):
    print(f"\n--- {t} " + "-" * max(0, 60 - len(t)))

print("=" * 70); print("core primitives -- invariant suite"); print("=" * 70)

section("purity")
check("0  no banned framework bound in our namespace", "PURITY" not in assert_pure(globals()))

rng = LCG(1234)

section("vector / matrix algebra")
a, b = randn_vector(6, rng), randn_vector(6, rng)
check("1  (a+b)-b == a",            *close(vsub(vadd(a, b), b), a))
check("2  a (*) 1 == a",            *close(vmul(a, [1.0]*6), a))
check("3  <a,b> == <b,a>",          *close([dot(a,b)], [dot(b,a)]))
check("4  <a,a> == ||a||^2",        *close([dot(a,a)], [l2(a)**2]))
check("5  ||a|| == sqrt(n)*rms(a)", *close([l2(a)], [math.sqrt(6)*rms(a)]))
A, B, Cm = randn_matrix(4,5,rng), randn_matrix(5,3,rng), randn_matrix(3,2,rng)
xv = randn_vector(5, rng)
check("6  matvec(M,x)[i] == <M[i],x>", *close(matvec(A,xv), [dot(A[i],xv) for i in range(4)]))
check("7a transpose(transpose(A)) == A", *close(transpose(transpose(A)), A))
check("7b (AB)^T == B^T A^T", *close(transpose(matmul(A,B)), matmul(transpose(B),transpose(A)), 1e-12, 1e-9))
check("7c A(BC) == (AB)C  [associativity]", *close(matmul(A,matmul(B,Cm)), matmul(matmul(A,B),Cm), 1e-12, 1e-9))
xv3 = randn_vector(3, LCG(7))
check("7d (AB)x == A(Bx)", *close(matvec(matmul(A,B),xv3), matvec(A,matvec(B,xv3)), 1e-12, 1e-9))

section("softmax and friends")
z = [1.0, -2.0, 0.5, 3.0, -0.25]; p = softmax(z)
check("8  softmax sums to 1",           *close([vsum(p)], [1.0]))
check("9  softmax strictly positive",   all(v > 0 for v in p))
check("10 shift invariance",            *close(softmax([v+17.3 for v in z]), p, 1e-12, 1e-9))
pb = softmax([1000.0, 999.0, 1001.0])
check("11 survives logits of 1e3",      abs(vsum(pb)-1.0) < 1e-12 and all(v==v for v in pb))
pm = softmax(apply_mask([1.0,2.0,3.0], [True,False,True]))
check("12 masked -> exactly 0.0",       pm[1] == 0.0 and abs(vsum(pm)-1.0) < 1e-12)
ls = log_softmax(z)
check("13a exp(log_softmax) == softmax", *close([math.exp(v) for v in ls], p))
check("13b log_softmax == log(softmax)", *close(ls, [math.log(v) for v in p]))
try:
    softmax([NEG_INF]*3); ok_allmask = False
except ValueError:
    ok_allmask = True
check("13c all-masked row raises, not NaN", ok_allmask)

section("nonlinearities")
check("14a sigma(0) == 0.5",                *close([sigmoid(0.0)], [0.5]))
check("14b sigma(-x) == 1 - sigma(x)",      *close([sigmoid(-2.7)], [1.0-sigmoid(2.7)]))
check("14c no overflow at +/-1000",         sigmoid(1000.0)==1.0 and sigmoid(-1000.0) < 1e-300)
check("14d sigma in [0,1] on a sweep",      all(0.0 <= sigmoid(v) <= 1.0 for v in [-50,-5,-1,0,1,5,50]))
check("15a gelu(0) == 0",                   gelu_exact(0.0)==0.0 and gelu_tanh(0.0)==0.0)
wg = max(abs(gelu_exact(-4+0.05*i)-gelu_tanh(-4+0.05*i)) for i in range(161))
check("15b tanh approx within 1e-2",        wg < 1e-2, f"max|diff|={wg:.3e}")
check("15c gelu dips below 0 for x<0",      gelu_exact(-0.5) < 0.0)

section("RMSNorm -- eq (2.9)")
vv = randn_vector(8, rng); r0 = rmsnorm_raw(vv, eps=0.0)
check("17a rms(output) == 1 at eps=0",      *close([rms(r0)], [1.0]))
check("17b does NOT centre (mean != 0)",    abs(mean(r0)) > 1e-6, f"mean={mean(r0):+.6f}")
check("17c scale invariant at eps=0",       *close(rmsnorm_raw(vscale(37.0,vv),eps=0.0), r0, 1e-12, 1e-9))
check("17d gain of ones is identity",       *close(rmsnorm(vv, gain=[1.0]*8, eps=0.0), r0))
rbig, rsml = rms(rmsnorm(vscale(1e2,vv),eps=1e-5)), rms(rmsnorm(vscale(1e-3,vv),eps=1e-5))
check("17e eps breaks invariance near 0",   rbig > 0.999 and rsml < 0.9,
      f"rms@1e2={rbig:.6f}  rms@1e-3={rsml:.6f}  <- the finding in 2.3")

section("shape plumbing")
yv = randn_vector(12, rng)
check("18a merge(split(x,H)) == x, H=1..12", all(merge_heads(split_heads(yv,H))==yv for H in [1,2,3,4,6,12]))
check("18b split gives H blocks of d/H",     shape(split_heads(yv,3)) == (3,4))
check("19a concat lengths add",              len(concat(a,b)) == len(a)+len(b))
check("19b concat puts e first",             concat([1.0,2.0],[9.0,9.0])[:2] == [1.0,2.0])

section("masks + claim C8")
check("20a causal row i allows i+1",  [sum(r) for r in causal_mask(4)] == [1,2,3,4])
cmk = causal_mask(4)
check("20b causal is lower-triangular", all(not cmk[i][j] for i in range(4) for j in range(4) if j>i))
for W in [1,2,3,8]:
    check(f"21 W={W}: row i allows min(W,i+1)",
          [sum(r) for r in sliding_window_mask(6,W)] == [min(W,i+1) for i in range(6)])
check("21b W=1 is the diagonal only",
      all(sliding_window_mask(5,1)[i][j] == (i==j) for i in range(5) for j in range(5)))
c8 = True
for W in [1,2,3,8]:
    m = sliding_window_mask(12, W)
    for t in range(11):
        rd  = {j for j in range(12) if m[t][j]}
        ret = {j for j in rd if j >= t-W+2}
        if ret | {t+1} != {j for j in range(12) if m[t+1][j]}: c8 = False
check("22 C8: retained(t) + current == read(t+1)", c8, "all W in {1,2,3,8}, all t")

section("attention")
q = randn_vector(4, rng)
ks = [randn_vector(4, rng) for _ in range(5)]
vs = [randn_vector(4, rng) for _ in range(5)]
o, sc, pr = attention(q, ks, vs)
check("23a probs sum to 1",                 *close([vsum(pr)], [1.0]))
check("23b scores == <q,k>/sqrt(d_head)",   *close(sc, [dot(q,k)/math.sqrt(4) for k in ks]))
check("23c out == sum_j p_j v_j by hand",   *close(o, [sum(pr[j]*vs[j][i] for j in range(5)) for i in range(4)]))
check("24 out inside convex hull of values",
      all(min(x[i] for x in vs)-1e-12 <= o[i] <= max(x[i] for x in vs)+1e-12 for i in range(4)))
check("25 identical keys -> uniform",       *close(attention(q,[ks[0]]*4,vs[:4])[2], [0.25]*4))
o1,_,p1 = attention(q,[ks[0]],[vs[0]])
check("26a single key -> p=1, out==value",  p1==[1.0] and allclose(o1, vs[0])[0])
os_,_,ps_ = attention(q,[vscale(80.0,q),ks[1],ks[2]],vs[:3])
check("26b dominant score -> one-hot",      ps_[0] > 1-1e-9 and allclose(os_, vs[0], 1e-8)[0])
mo1,ms1,_ = multihead_attention(split_heads(q,1), [split_heads(k,1) for k in ks], [split_heads(x,1) for x in vs])
check("27a H=1 reduces to single-head",     *close(mo1, o))
check("27b scores shape [H][n_keys]",       shape(ms1) == (1,5))
q2 = randn_vector(8, rng); k2=[randn_vector(8,rng) for _ in range(3)]; v2=[randn_vector(8,rng) for _ in range(3)]
mo2,ms2,mp2 = multihead_attention(split_heads(q2,2),[split_heads(k,2) for k in k2],[split_heads(x,2) for x in v2])
check("27c each head's probs sum to 1",     all(abs(vsum(r)-1.0) < 1e-12 for r in mp2))
check("27d heads do not mix",               *close(mo2[:4], attention(split_heads(q2,2)[0],
          [split_heads(k,2)[0] for k in k2], [split_heads(x,2)[0] for x in v2])[0]))
check("27e scale is 1/sqrt(d_head) not 1/sqrt(d)",
      *close([ms2[0][0]], [dot(split_heads(q2,2)[0], split_heads(k2[0],2)[0])/2.0]))

section("random generator")
ga2, gb2, gc2 = LCG(99), LCG(99), LCG(100)
s1 = [ga2.normal() for _ in range(5)]; s2 = [gb2.normal() for _ in range(5)]; s3 = [gc2.normal() for _ in range(5)]
check("28a same seed -> same stream",  s1 == s2)
check("28b different seeds differ",    s1 != s3)
check("28c stream does not repeat",    len(set(s1)) == 5)
gN = LCG(2024); smp = [gN.normal() for _ in range(20000)]
mu_, sd_ = mean(smp), math.sqrt(var(smp))
check("29 Box-Muller mean~0 std~1",    abs(mu_) < 0.05 and abs(sd_-1.0) < 0.05, f"mean={mu_:+.4f} std={sd_:.4f}")
g5b = LCG(5); uu = [g5b.uniform() for _ in range(1000)]
bn = [0]*10
for t in uu: bn[min(9,int(t*10))] += 1
check("30a uniform strictly inside (0,1)", all(0.0 < t < 1.0 for t in uu))
check("30b uniform actually varies",       len(set(uu)) > 990, f"{len(set(uu))}/1000 distinct")
check("30c 10 bins within 100+/-40",       all(60 <= c <= 140 for c in bn), f"{bn}")

section("inspection helpers")
check("31a shape(scalar) == ()",       shape(3.0) == ())
check("31b shape nests [H][n]",        shape([[1.0,2.0],[3.0,4.0],[5.0,6.0]]) == (3,2))
rec = summarize("s_t", [3.0,-1.0,0.0,4.0])
check("31c summarize stats correct",   rec["shape"]==[4] and rec["min"]==-1.0 and rec["max"]==4.0
      and abs(rec["mean"]-1.5)<1e-12 and abs(rec["l2"]-math.sqrt(26))<1e-12)
check("31d counts masked entries",     summarize("m",[1.0,NEG_INF,2.0])["n_masked"] == 1)
okd, wd, idx = allclose([1.0,2.0],[1.0,2.5])
check("31e allclose reports worst index", (not okd) and idx == 1)

print("\n" + "=" * 70)
print(f"PASSED {len(_PASS)}   FAILED {len(_FAIL)}")
for n in _FAIL: print("   FAILED:", n)
print("all primitives hold." if not _FAIL else "SUITE RED")
print("=" * 70)

# Where we are, and what comes next

## ✅ Phases 0–2 complete

**Phase 0** — the paper extracted: notation, all 16 equations verbatim, four stores, both propositions, the BPTT appendix, and a complete list of what is unspecified with our labelled assumptions.

**Phase 1** — the computation graph, every step as INPUT → OPERATION → OUTPUT → SHAPE → MEANING, with the recurrent path drawn unrolled and **both** channels visible.

**Phase 2** — every primitive, built from `math` alone. **67 invariants, 67 passing.**

---

## Three things we learned before writing a single line of model code

**1. The paper reports no experiments.** So "reproduction" here means *verifying propositions and structural claims exactly*, not re-running a benchmark. And the width-512 / FFN-1365 / 4+4…8+0 config often attributed to this paper is not in it — it is an earlier reproduction's design. (§0, top.)

**2. There are two recurrent channels, not one.** Appendix B's $2\times2$ Jacobian says the state crossing each token boundary is $(s_t, C^D_t)$ — the vector **and** every layer's sliding window. This has a direct consequence we have already folded into the plan: **$\alpha=0$ is not a non-recurrence ablation.** It severs channel 1 and leaves channel 2 running. Phase 7 is therefore a **2×2 over $(\alpha, W)$**, with $\alpha=0 \wedge W=1$ as the only genuine non-recurrent control.

**3. RMSNorm's $\epsilon$ is not scale-invariant near zero** — output RMS falls to 0.457 at $\mathrm{rms}(x)=1.6\times10^{-3}$, and to 0.051 at $1.6\times10^{-4}$. A contracting state keeps shrinking *through* eq (2.9), which scales down the whole feedback term. That is a concrete mechanism behind the paper's own §3.3 caveat. Whether $\|s_t\|$ actually enters that regime is a **measurement for Phase 6**, not an assumption.

And **claim C8 is verified**: the read window $[\max(1,t{-}W{+}1), t]$ and the retention rule $[\max(1,t{-}W{+}2), t]$ differ by one, which reads like an off-by-one, but `retained(t) ∪ {t+1} == read_window(t+1)` holds exactly for every $W$ and every $t$.

---

## ▶ Phase 3 — the tiny RLT

Next cell block builds the model at a size where **every number fits on screen**:

```
V = 10    d = 8    L_E = 1    L_D = 1    H = 2    d_head = 4
d_ff = 16    W = 3    G = 1    alpha = 0.1
```

Components, in the order the graph needs them:

`TokenEmbedding` → `CausalEncoder` → `EncoderMemory` → `GatedMerge` → `DecoderSWA` → `CrossAttention` → `FFN` → `DecoderBlock` → `RLT` → `Readout`

Each one gets a numerical example printed immediately after it is defined, and an invariant test, before we move on. No component goes in unverified.

**Then Phase 4** makes the model talk: a full forward trace on `[BOS, 3, 7, 2]` printing, per token — the embedding, every Q/K/V, every attention score and probability, the SWA cache **before and after** with evicted entries named, $s_{t-1}$, $\mathrm{RMSNorm}(s_{t-1})$, $[e_t; r_{t-1}]$, the gate preactivation, $g_t$, $W_s r_{t-1}$, the feedback contribution, $u_t$, every layer output, $s_t$, the logits, and the prediction.

---

### Open questions this dissection exists to answer

1. Does $s_t$ carry information, or does the SWA cache do all the work? (B.3 says both channels exist — **measure the split**.)
2. At what $t$ does the influence of $x_1$ on $s_t$ fall below float noise?
3. What does $g_t$ do numerically — saturated, near-constant, or genuinely input-dependent?
4. Does $\|\prod J_t\|$ grow or contract, and does the $s$-only product differ from the full product as C9 claims?
5. Is the $\alpha=0$ control actually a non-recurrent model? **Predicted: no.**

None of these are answerable yet. That is the point of stopping here.

In [ ]:
#@title  utility: sync this notebook -> GitHub  (source-only, outputs stripped)
# Not part of the dissection. Keeps the repo mirror current after each phase.
# Token comes from Colab Secrets (key icon, left sidebar) -> name it GITHUB_TOKEN.
# Nothing secret is ever printed or written into the notebook.

import json, os, subprocess, textwrap

REPO = "Maverick-Ansh/recurrent_looped_transformer_scratch"
NB_NAME = "RLT_dissection.ipynb"

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as e:
    TOKEN = None
    print("no GITHUB_TOKEN available:", type(e).__name__, "-", str(e)[:200])

from google.colab import _message
nb = _message.blocking_request("get_ipynb", timeout_sec=120)["ipynb"]

# strip outputs + execution counts: the repo copy is source-only
for c in nb.get("cells", []):
    if c.get("cell_type") == "code":
        c["outputs"] = []
        c["execution_count"] = None
n_code = sum(1 for c in nb["cells"] if c["cell_type"] == "code")
n_md   = sum(1 for c in nb["cells"] if c["cell_type"] == "markdown")
print(f"notebook: {len(nb['cells'])} cells ({n_code} code, {n_md} markdown)")

if not TOKEN:
    os.makedirs("/content/out", exist_ok=True)
    with open(f"/content/out/{NB_NAME}", "w") as f:
        json.dump(nb, f, indent=1)
    print(f"\nwrote /content/out/{NB_NAME} -- add GITHUB_TOKEN to Colab Secrets to push,")
    print("or download it from the file browser on the left.")
else:
    subprocess.run("rm -rf /content/_sync", shell=True)
    r = subprocess.run(f"git clone -q https://{TOKEN}@github.com/{REPO}.git /content/_sync",
                       shell=True, capture_output=True, text=True)
    if r.returncode:
        print("clone failed:", r.stderr[-400:])
    else:
        with open(f"/content/_sync/{NB_NAME}", "w") as f:
            json.dump(nb, f, indent=1)
        cmds = [
            'git -C /content/_sync config user.email "anshvivek2003@gmail.com"',
            'git -C /content/_sync config user.name  "Maverick-Ansh"',
            'git -C /content/_sync add -A',
            'git -C /content/_sync commit -q -m "Sync Colab notebook (Phases 0-2)\n\n'
            'Co-Authored-By: Claude Opus 5 <noreply@anthropic.com>" || echo "nothing to commit"',
            'git -C /content/_sync push -q origin main',
        ]
        for c in cmds:
            rr = subprocess.run(c, shell=True, capture_output=True, text=True)
            if rr.returncode and "nothing to commit" not in (rr.stdout + rr.stderr):
                print("FAILED:", c.split("-C /content/_sync")[-1][:60], "->", (rr.stderr or rr.stdout)[-300:])
                break
        else:
            print(f"pushed {NB_NAME} to https://github.com/{REPO}")